# Regularization

We are going to look at a dataset on baseball to see if we can predict a player's salary.


## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler


## Read in Data

In [ ]:
url = "http://ddc-datascience.s3.amazonaws.com/Hitters.csv"
hits = pd.read_csv( url )
hits.head()

In [ ]:
hits.shape

In [ ]:
hits.info()

In [ ]:
hits.describe(include = "all").transpose()

## Data Cleaning

### Backup

In [ ]:
# Backup
hits_clean = hits.copy()
hits_clean


In [ ]:
# Restore
hits = hits_clean.copy()
hits


### Check for nulls


In [ ]:
# Check for NAs
hits_clean.isnull().sum()*1000


In [ ]:
nulls = hits_clean.isnull().sum()
nulls


In [ ]:
filter = nulls > 0
filter

In [ ]:
nulls[ filter ]/(hits_clean.shape[0])*100

All the NAs are in the Salary column. Since the salary column is our response, we need to remove rows that contain NAs.

In [ ]:
# Remove rows with NAs
hits_clean.dropna(
  subset = ["Salary"],
  axis = 0,
  inplace = True,
)

In [ ]:
pd.DataFrame.dropna?

In [ ]:
# Confirm they were removed
hits_clean.info()

In [ ]:
hits_clean.isnull().sum()*1000

In [ ]:
nulls = hits_clean.isnull().sum()
filter = nulls > 0
nulls[ filter ]/(hits_clean.index.size)*100

NAs were successfully removed. Now we notice that some of the columns are object type. Let's take a look at those.

In [ ]:
hits_clean.dtypes.value_counts()

In [ ]:
hits_clean.dtypes.sort_values()

### Backup

In [ ]:
# make a backup
hits_clean_bak = hits_clean.copy()
hits_clean_bak

In [ ]:
# restore from backup
hits_clean = hits_clean_bak.copy()
hits_clean

### League

In [ ]:
# League column
hits_clean['League'].value_counts()

In [ ]:
# Since it only has two values, we can label encode
hits_clean['League'] = hits_clean['League'].replace({'A': "1", 'N': "0"}).astype(int)

In [ ]:
hits_clean['League'].value_counts()

### Division

In [ ]:
# Division column
hits_clean['Division'].value_counts()

In [ ]:
# Since it only has two values, we can label encode
hits_clean['Division'] = hits_clean['Division'].replace({'W': "1", 'E': "0"}).astype(int)

In [ ]:
hits_clean['Division'].value_counts()

### New League

In [ ]:
# New League column
hits_clean['NewLeague'].value_counts()

In [ ]:
# Since it only has two values, we can label encode
hits_clean['NewLeague'] = hits_clean['NewLeague'].replace({'A': "1", 'N': "0"}).astype(int)

In [ ]:
hits_clean['NewLeague'].value_counts()

In [ ]:
hits_clean.head()

In [ ]:
hits_clean.info()

In [ ]:
hits_clean.dtypes.sort_values()

## EDA

### Correlation matrix plot


In [ ]:
# Let's take a look at a correlation plot
plt.figure(figsize=(15,15))
correlation_matrix = hits_clean.corr().round(2)
sns.heatmap(
    data=correlation_matrix,
    annot=True,
    vmin=-1,
    vmax=1,
) ;

### Correlation matrix plot - lower triangle


In [ ]:
np.set_printoptions(linewidth=200, suppress=True)

ones = np.ones_like(
    correlation_matrix,
    dtype=bool,
)
ones


In [ ]:
# Create a mask
mask = np.tril(ones, k=-1)
mask


In [ ]:
lower_triangle_corr = correlation_matrix.where(mask)
lower_triangle_corr


In [ ]:
plt.figure(figsize=(15,15))
sns.heatmap(
    data=lower_triangle_corr,
    annot=True,
    vmin=-1,
    vmax=1,
) ;

Create a list of the most highly correlated.

In [ ]:
lt_df = (
  lower_triangle_corr
  .abs()
  .unstack()
  .to_frame()
  .rename( columns={0:"A"} )
  .query( "A > 0.90")
  .sort_values( by = "A", ascending=False )
  .reset_index()
)
lt_df


In [ ]:
# get list of highly correlated features
list(enumerate(set(lt_df.iloc[:,:2].stack().sort_values().to_list())))


Several of the predictors are highly correlated and cluster into two groups:
- Years, CAtBat, CHits, CHmRun, CRuns, CRBI, CWalks
- AtBat, Hits, Runs

In [ ]:
# Let's look at the distribution of salaries
plt.hist(hits_clean['Salary'])
plt.xlabel('Salary in Thousands')
plt.ylabel("Count")
plt.savefig('test.png')

Within each group, the features that are most highly correlated with salary are CRBI and Hits.

In [ ]:
# Let's look at a scatterplot of CRBI and Salary
hits_clean.plot('CRBI', 'Salary', kind = 'scatter') ;

In [ ]:
# Let's look at a scatterplot of hits and salary
hits_clean.plot('Hits', 'Salary', kind = 'scatter') ;

## Linear Regression
We will first fit a multiple linear regression model. We are going to drop some of the highly correlated predictors and keep the two most highly correlated with Salary: Hits and CRBI.

In [ ]:
# First we will break up our data into training and testing sets
# X = hits_clean.drop(['CRuns', 'CHmRun', 'CHits', 'CAtBat', 'CWalks', 'Runs', 'AtBat', 'Years', 'Salary'], axis = 1).copy()

trimmed_df = hits_clean[['Salary', 'AtBat', 'Hits', 'Years']].copy()
# trimmed_df = hits_clean[['Salary', 'Hits', 'Years']].copy()
# trimmed_df = hits_clean[['Salary', 'AtBat', 'Years']].copy()
trimmed_df


In [ ]:
X = trimmed_df.drop(columns = ['Salary'])
y = trimmed_df['Salary']

# normalize the features
scaler = StandardScaler()
scaler.fit_transform(X)


In [ ]:
X.columns


In [ ]:
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size = 0.2, random_state=5)


In [ ]:
X.head()


In [ ]:
X_scaled.head()


In [ ]:
X_scaled.describe().transpose()


In [ ]:
# Fit a linear model using Sklearn
model = LinearRegression()
my_fit_lr = model.fit(X_train, y_train)


In [ ]:
# Create a data frame with the results from the linear regression models
model_results = pd.DataFrame()
model_results['Predictor'] = X_train.columns
model_results['Linear'] = my_fit_lr.coef_
model_results.sort_values(by=['Linear'],inplace=True, ascending=False)
model_results


In [ ]:
# We can even plot the coefficients if we want to!
model_results.plot(x = 'Predictor', y = 'Linear', kind = 'bar')
plt.grid()


In [ ]:
# Calculate CV RMSE
results = cross_val_score(
    model,
    X_scaled,
    y,
    scoring='neg_root_mean_squared_error',
    cv = 10
    )
rmse = abs(results.mean())
print(f"CV RMSE: {rmse}")


- both: CV RMSE: 359.4996029118099
- with Hits: CV RMSE: 358.6910385178394
- with AtBats: CV RMSE: 366.1313104847557


This is showing that we have a RMSE of about $360k.

**NOTE**: We did not do any residual analysis in this example, although it is an important part of checking the assumptions of a linear regression model. Please refer back to the linear regression lecture for an example of how to check residuals.

## Regularization - Ridge Regression

Ridge regression is ideal for preventing overfitting and stabilizing model coefficients when your dataset suffers from multicollinearity. Instead of dropping correlated features, it shrinks their weights proportionally and keeps them all in the model. Use it when you want reliable out-of-sample predictions without having to manually prune redundant variables.

However, in this example, we'll use the trimmed data frame from the Linear Regression for comparison.


In [ ]:
# Train the model
my_fit_rr = Ridge(alpha = 1) # Arbitrarily choosing alpha = 1
my_fit_rr.fit(X_train, y_train) ;

In [ ]:
# Calculate RMSE
y_pred = my_fit_rr.predict(X_test)
rmse = np.sqrt(np.mean(np.square(y_test - y_pred)))
print(f"RMSE: {rmse}")

Let's try looking at different values of $\alpha$.

In [ ]:
# Specify values of alpha we want to try
alpha = [1e-15, 1e-10, 1e-8, 1e-4, 1e-3, 1e-2, 1e-1, 1, 2, 3, 4, 5, 10, 20, 30, 50]

# Create empty list to hold results
results_alpha = []

for a in alpha:
  # Train the model
  my_fit_rr = Ridge(alpha = a) # Looping through alpha
  my_fit_rr.fit(X_train, y_train)

  # Calculate CV RMSE
  results = cross_val_score(
      my_fit_rr,
      X_scaled,
      y,
      scoring='neg_root_mean_squared_error',
      cv = 10
      )
  rmse = abs(results.mean())
  results_alpha.append(rmse)

results_df = pd.DataFrame({'Alpha': alpha, 'RMSE': results_alpha})
print(results_df.sort_values(by=['RMSE']))

In [ ]:
results_df.sort_values( by = ["Alpha"])

In [ ]:
results_df.sort_values( by = ["Alpha"])["RMSE"].plot()

It looks like 20 gives us the smallest RMSE.

In [ ]:
# Retrain the model w/ alpha = 20
my_fit_rr = Ridge(alpha = 30)
my_fit_rr.fit(X_train, y_train)

# Calculate CV RMSE
results = cross_val_score(my_fit_rr, X_scaled, y, scoring='neg_root_mean_squared_error', cv = 10)
rmse = abs(results.mean())
print(f"CV RMSE: {rmse}")

In [ ]:
# Add ridge regression results to data frame we created earlier
model_results['Ridge'] = my_fit_rr.coef_

In [ ]:
model_results

## Regularization - Lasso Regression


Lasso regression is ideal when you want automated feature selection, as its L1 penalty can shrink coefficient values all the way down to zero. This makes it exceptionally useful for high-dimensional datasets where you need to filter out noise and identify the most critical predictors. Ultimately, it produces simpler, highly sparse models that are much easier to interpret and less prone to overfitting.


In [ ]:
# Specify values of alpha we want to try
alpha = [1e-15, 1e-10, 1e-8, 1e-4, 1e-3, 1e-2, 1e-1, 1, 2, 3, 4, 5, 10, 20, 30, 50]

# Create empty list to hold results
results_alpha = []

for a in alpha:
  # Train the model
  my_fit_lasso = Lasso(alpha = a, tol = .01, max_iter=100_000) # Looping through alpha
  my_fit_lasso.fit(X_train, y_train)

  # Calculate CV RMSE
  results = cross_val_score(
      my_fit_lasso,
      X_scaled,
      y,
      scoring='neg_root_mean_squared_error',
      cv = 10
  )
  rmse = abs(results.mean())
  results_alpha.append(rmse)

results_df = pd.DataFrame({'Alpha': alpha, 'RMSE': results_alpha})
print(results_df.sort_values(by=['RMSE']))

In [ ]:
results_df

In [ ]:
# Train the model
my_fit_lasso = Lasso(alpha = 20, tol = .01, max_iter=1_000_000)
my_fit_lasso.fit(X_train, y_train)

# Calculate CV RMSE
results = cross_val_score(my_fit_lasso, X_scaled, y, scoring='neg_root_mean_squared_error', cv = 10)
rmse = abs(results.mean())
print(f"CV RMSE: {rmse}")

In [ ]:
# Add Lasso results to the data frame we created before
model_results['Lasso'] = my_fit_lasso.coef_

## Summary

In [ ]:
model_results

In [ ]:
# Plot results
fig, ax = plt.subplots(1,1, figsize = (10,7))
model_results.plot(x = 'Predictor', kind = 'bar', ax = ax)
ax.set_ylabel('Coefficient')
plt.grid()

After performing regularization, we were able to reduce our RMSE from
\$360k with linear regression to
\$329k with Ridge regression to
\$327k with Lasso regression.

For all models, CRBI seems to be one of the more important predictors of salary.

